# Modelling: linear regression pipeline tracked with MLflow

Goal: predict the age of an abalone (`Age = Rings + 1.5`) from its physical measurements.

This notebook builds on [eda.ipynb](eda.ipynb): the same data path and target, the data-quality rules, and the encoding of `Sex` all come from its conclusions.

Guidelines:
- A simple linear regression model is enough.
- Experiments are tracked with MLflow; compare them in the MLflow UI (`mlflow ui --backend-store-uri sqlite:///mlflow.db` from the project root).
- MLflow data is **not** pushed to the repository (`mlflow.db` and `mlruns/` are git-ignored).

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 500)

## Data

In [ ]:
DATA_PATH = "../data/abalone.csv"
TARGET = "Rings"
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
df.head()

Following the data-quality checks of the EDA, rows with physically impossible values are removed before modelling: a shell `Height` of 0, or a part (shucked, viscera or shell) heavier than the whole abalone.

In [ ]:
invalid_mask = (
    (df["Height"] == 0)
    | (df["Shucked weight"] > df["Whole weight"])
    | (df["Viscera weight"] > df["Whole weight"])
    | (df["Shell weight"] > df["Whole weight"])
)
print("Rows removed (impossible values):", int(invalid_mask.sum()))
df = df[~invalid_mask].reset_index(drop=True)

# Age in years: the number of shell rings + 1.5 (standard convention for abalone)
df["Age"] = df[TARGET] + 1.5
X = df.drop(columns=[TARGET, "Age"])
y = df["Age"]

print("Final shape:", X.shape)
df["Age"].describe()

## Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## Pipeline

One-hot encode the categorical `Sex` feature (the EDA showed that infants differ clearly from adults), standardize the numerical features, then fit a linear regression. The numerical features are strongly correlated with each other (EDA correlation matrix), which makes individual coefficients hard to interpret, but predictions remain usable. Putting the preprocessing inside the pipeline means the scaler and encoder are fitted on the training data only, and the whole model is logged to MLflow as a single artifact.

In [ ]:
categorical_features = ["Sex"]
numerical_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numerical", StandardScaler(), numerical_features),
    ]
)

linear_regression_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

## MLflow experiment

Each run logs its parameters, cross-validated and test metrics (RMSE, MAE and R² in years), and the fitted model. A mean-prediction baseline is logged in the same experiment so the linear regression can be judged against it.

In [ ]:
MLFLOW_TRACKING_URI = "sqlite:///../mlflow.db"  # local database, kept out of git
EXPERIMENT_NAME = "abalone-age-prediction"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Artifacts (models, metadata) go to the project-level mlruns/ folder, not notebooks/mlruns/
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(
        EXPERIMENT_NAME, artifact_location=Path("../mlruns").resolve().as_uri()
    )
mlflow.set_experiment(EXPERIMENT_NAME)


def evaluate(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_eval, y_pred))),
        "mae": float(mean_absolute_error(y_eval, y_pred)),
        "r2": float(r2_score(y_eval, y_pred)),
    }


def run_experiment(run_name, model, params):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("random_state", RANDOM_STATE)

        cv_rmse = -cross_val_score(
            model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
        )
        mlflow.log_metric("cv_rmse_mean", float(cv_rmse.mean()))
        mlflow.log_metric("cv_rmse_std", float(cv_rmse.std()))

        model.fit(X_train, y_train)
        metrics = evaluate(model, X_test, y_test)
        mlflow.log_metrics({f"test_{name}": value for name, value in metrics.items()})

        mlflow.sklearn.log_model(
            model,
            name="model",
            signature=infer_signature(X_train, model.predict(X_train)),
            input_example=X_test.head(5),
        )

    print(
        run_name,
        "| CV RMSE:",
        round(cv_rmse.mean(), 3),
        "| test:",
        {k: round(v, 3) for k, v in metrics.items()},
    )
    return model, metrics

In [ ]:
baseline_model, _ = run_experiment(
    run_name="baseline-mean",
    model=DummyRegressor(strategy="mean"),
    params={"model_type": "DummyRegressor", "strategy": "mean"},
)

In [ ]:
linear_model, linear_metrics = run_experiment(
    run_name="linear-regression",
    model=linear_regression_pipeline,
    params={
        "model_type": "LinearRegression",
        "categorical_features": str(categorical_features),
    },
)

## Results

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["start_time DESC"],
)
runs[
    [
        "tags.mlflow.runName",
        "metrics.cv_rmse_mean",
        "metrics.test_rmse",
        "metrics.test_mae",
        "metrics.test_r2",
    ]
]

In [ ]:
y_pred = linear_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.4, s=12)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", label="perfect prediction")
ax.set_xlabel("True age (years)")
ax.set_ylabel("Predicted age (years)")
ax.set_title("Linear regression: predicted vs true age (test set)")
ax.legend()
plt.show()

## Conclusions

- The linear regression clearly beats the mean baseline: test RMSE drops from about 3.34 to 2.17 years, MAE from 2.45 to 1.53 years, and R² rises from 0 to 0.58.
- The 5-fold CV RMSE (2.24) is close to the test RMSE (2.17), so the model is stable and not overfitting.
- The scatter plot shows the main limitation, as anticipated in the EDA: predictions are compressed toward the middle of the range. Abalones older than about 15 years are systematically under-predicted, and very young ones are slightly over-predicted. Those older abalones are also rare in the data, so they weigh little in the metrics. A linear model cannot capture this non-linearity; a tree-based model or a log-transformed target are natural next experiments to log in MLflow.
- The cleaning rules from the EDA removed 6 rows with impossible values before training.